# btc_price — evolutionary BTC price predictor (1 min → 12 h)

End-to-end, self-contained notebook.

| Step | What happens |
|---|---|
| 0 | Setup and configuration |
| 1 | **Compute detection** — GPUs, **NVLink**, cores, RAM → execution plan |
| 2 | Feature engineering (57 causal features, 9 horizons) |
| 3 | Evolvable model zoo (MLP / GRU / TCN / Transformer) |
| 4 | Genetic operators (tournament, BLX crossover, mutation, elitism) |
| 5 | **Island-model evolutionary training** — GPU + CPU islands, resumable, 5 h |
| 6 | Held-out evaluation |
| 7 | Forecast 1 min → 12 h |
| 8 | Publish to GitHub |

The dataset is produced by `src/download_data.py` (one year of 1-minute
BTCUSDT OHLCV from Binance's public dumps) and is **not** part of this
notebook — run that script once, or pull `data/processed/btc_1m.parquet`
from the repo.

**Targets are log returns, not raw prices.** Predicting price directly
degenerates into echoing the last close and looks deceptively accurate.
A skill score below 1.0 means the model genuinely beats "assume no change".

## 0 · Setup

In [ ]:
import os, sys, json, math, time, pickle, random, signal, hashlib, copy, warnings
import argparse, shutil, subprocess, mimetypes
import urllib.request, urllib.error
from pathlib import Path
from dataclasses import dataclass, field, asdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from contextlib import ExitStack

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Repo root: works whether the notebook sits in notebooks/ or at the top level
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
CACHE = ROOT / "data" / "processed" / "cache"
CKPT  = Path(os.environ.get("BTC_CKPT_DIR", ROOT / "checkpoints"))
LOGS  = ROOT / "logs"
DATA  = ROOT / "data" / "processed" / "btc_1m.parquet"
for d in (CACHE, CKPT, CKPT / "elites", LOGS):
    d.mkdir(parents=True, exist_ok=True)

_STOP = {"flag": False}
warnings.filterwarnings("ignore", category=UserWarning)

print("root   :", ROOT)
print("torch  :", torch.__version__)
print("dataset:", DATA, "-", "FOUND" if DATA.exists() else "MISSING (run src/download_data.py)")

## 1 · System detection — GPUs and NVLink

Decides how the evolution is parallelised:

| Detected | Strategy |
|---|---|
| ≥2 GPUs **with NVLink** | `ddp-nvlink` — NCCL with P2P over NVLink, one island per GPU |
| ≥2 GPUs, PCIe only | `ddp-pcie` — NCCL, P2P disabled |
| 1 GPU | concurrent training instances sharing the device |
| 0 GPUs | parallel CPU worker processes |

In every case the population is evaluated in parallel and the **best
individual seeds the next generation**, which is exactly the "run multiple
training instances and pick the best" fallback.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import time
from dataclasses import asdict, dataclass, field


def _run(cmd: list[str], timeout: int = 20) -> str | None:
    if not shutil.which(cmd[0]):
        return None
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return p.stdout if p.returncode == 0 else None
    except (subprocess.TimeoutExpired, OSError):
        return None


# --------------------------------------------------------------------------- #
# dataclasses                                                                  #
# --------------------------------------------------------------------------- #
@dataclass
class GPUInfo:
    index: int
    name: str
    memory_mb: int
    free_mb: int = 0
    compute_capability: str = "?"
    multi_processors: int = 0
    supports_tf32: bool = False
    supports_bf16: bool = False


@dataclass
class ExecutionPlan:
    gpu_islands: int = 0
    islands_per_gpu: int = 0
    cpu_islands: int = 0
    total_islands: int = 1
    threads_per_cpu_island: int = 1
    threads_per_gpu_island: int = 1
    precision: str = "fp32"          # bf16 | fp16 | fp32
    data_placement: str = "mmap"     # vram | ram | mmap
    pin_memory: bool = False
    gpu_mem_budget_mb: int = 0
    cpu_mem_budget_mb: int = 0
    use_tf32: bool = False
    dataset_mb: float = 0.0
    notes: list[str] = field(default_factory=list)


@dataclass
class SystemInfo:
    # GPU
    gpu_count: int = 0
    gpus: list[GPUInfo] = field(default_factory=list)
    nvlink_available: bool = False
    nvlink_active_links: int = 0
    nvlink_pairs: list[tuple[int, int]] = field(default_factory=list)
    nvlink_fully_connected: bool = False
    nvlink_bandwidth_gbs: float = 0.0
    p2p_matrix: list[list[bool]] = field(default_factory=list)
    topology_matrix: str = ""
    driver_version: str = ""
    cuda_version: str = ""
    mps_available: bool = False
    # CPU / memory
    cpu_logical: int = 0
    cpu_physical: int = 0
    numa_nodes: int = 1
    cpu_model: str = ""
    cpu_flags: list[str] = field(default_factory=list)
    total_ram_gb: float = 0.0
    avail_ram_gb: float = 0.0
    swap_gb: float = 0.0
    disk_free_gb: float = 0.0
    shm_total_mb: float = 0.0
    shm_free_mb: float = 0.0
    blockers: list[str] = field(default_factory=list)
    # software
    torch_version: str = "not installed"
    cuda_available: bool = False
    nccl_available: bool = False
    # derived
    strategy: str = "cpu-islands"
    world_size: int = 1
    plan: ExecutionPlan = field(default_factory=ExecutionPlan)
    notes: list[str] = field(default_factory=list)

    def to_dict(self):
        d = asdict(self)
        d["gpus"] = [asdict(g) for g in self.gpus]
        d["plan"] = asdict(self.plan)
        return d


# --------------------------------------------------------------------------- #
# NVLink / P2P                                                                 #
# --------------------------------------------------------------------------- #
def detect_nvlink(gpu_count: int) -> dict:
    out = {"available": False, "active_links": 0, "pairs": [],
           "fully_connected": False, "topology": "", "bandwidth": 0.0}
    if gpu_count < 1:
        return out

    status = _run(["nvidia-smi", "nvlink", "--status"])
    if status:
        bw = [float(x) for x in re.findall(r"Link\s+\d+:\s+([\d.]+)\s*GB/s", status)]
        out["active_links"] = len(bw)
        out["available"] = len(bw) > 0
        out["bandwidth"] = round(sum(bw) / len(bw), 2) if bw else 0.0

    topo = _run(["nvidia-smi", "topo", "-m"])
    if topo:
        out["topology"] = topo.strip()
        pairs, nvl = [], 0
        for line in topo.splitlines():
            m = re.match(r"^\s*GPU(\d+)\s+(.*)$", line)
            if not m:
                continue
            i, cells = int(m.group(1)), m.group(2).split()[:gpu_count]
            for j, c in enumerate(cells):
                if re.fullmatch(r"NV\d+", c):
                    nvl += 1
                    if i < j:
                        pairs.append((i, j))
        out["pairs"] = sorted(pairs)
        if nvl:
            out["available"] = True
        out["fully_connected"] = (gpu_count > 1
                                  and len(pairs) == gpu_count * (gpu_count - 1) // 2)
    return out


def _p2p_matrix(n: int) -> list[list[bool]]:
    try:
        import torch
        return [[bool(i != j and torch.cuda.can_device_access_peer(i, j))
                 for j in range(n)] for i in range(n)]
    except Exception:
        return []


# --------------------------------------------------------------------------- #
# CPU / memory / disk                                                          #
# --------------------------------------------------------------------------- #
def _cpu_details(info: SystemInfo) -> None:
    info.cpu_logical = os.cpu_count() or 1
    info.cpu_physical = info.cpu_logical

    txt = _run(["lscpu"]) or ""
    def grab(pat, cast=str, default=None):
        m = re.search(pat, txt)
        return cast(m.group(1).strip()) if m else default

    cps = grab(r"Core\(s\) per socket:\s+(\d+)", int)
    sockets = grab(r"Socket\(s\):\s+(\d+)", int)
    if cps and sockets:
        info.cpu_physical = cps * sockets
    info.numa_nodes = grab(r"NUMA node\(s\):\s+(\d+)", int, 1) or 1
    info.cpu_model = grab(r"Model name:\s+(.+)", str, "") or ""

    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("flags"):
                    all_flags = set(line.split(":", 1)[1].split())
                    want = ["avx2", "avx512f", "avx512_bf16", "avx_vnni",
                            "amx_bf16", "amx_int8", "sha_ni", "f16c"]
                    info.cpu_flags = [w for w in want if w in all_flags]
                    break
    except Exception:
        pass

    try:
        with open("/proc/meminfo") as f:
            mi = f.read()
        g = lambda k: int(re.search(rf"{k}:\s+(\d+)", mi).group(1)) / 1024 / 1024
        info.total_ram_gb = round(g("MemTotal"), 2)
        info.avail_ram_gb = round(g("MemAvailable"), 2)
        info.swap_gb = round(g("SwapTotal"), 2)
    except Exception:
        pass

    try:
        s = os.statvfs(os.path.dirname(os.path.abspath(__file__)))
        info.disk_free_gb = round(s.f_bavail * s.f_frsize / 1e9, 2)
    except Exception:
        pass
    try:
        s = os.statvfs("/dev/shm")
        info.shm_total_mb = round(s.f_blocks * s.f_frsize / 1e6, 1)
        info.shm_free_mb = round(s.f_bavail * s.f_frsize / 1e6, 1)
    except Exception:
        pass


# --------------------------------------------------------------------------- #
# planner                                                                      #
# --------------------------------------------------------------------------- #
GPU_RUNTIME_MB = 900        # CUDA context + torch per process
CPU_RUNTIME_MB = 380        # torch + numpy RSS for the first process
CUDA_HOST_MB = 1400         # host RSS per process holding a CUDA context
SHM_PER_PROC_MB = 256       # /dev/shm a torch worker wants
CPU_FORK_MARGINAL_MB = 260  # extra RSS per FORKED island (shares .so pages)
MIN_ACT_MB = 64             # smallest useful activation budget


def build_plan(info: SystemInfo, dataset_mb: float = 160.0,
               max_islands_per_gpu: int = 4,
               allow_cpu_islands: bool = True,
               force_cpu_islands: int | None = None) -> ExecutionPlan:
    p = ExecutionPlan(dataset_mb=dataset_mb)

    # ---- GPU islands: pack as many per card as VRAM allows ---------------- #
    if info.gpu_count:
        free = min((g.free_mb or g.memory_mb) for g in info.gpus)
        # each island needs: runtime + a resident copy of the dataset + activations
        per_island = GPU_RUNTIME_MB + dataset_mb + 400
        n = int(max(1, min(max_islands_per_gpu, free // max(per_island, 1))))
        # every CUDA worker is a process holding a context: it needs shared
        # memory and host RAM, and those run out long before VRAM does
        if info.shm_free_mb > 0:
            by_shm = max(1, int(info.shm_free_mb // SHM_PER_PROC_MB))
            if by_shm < n * info.gpu_count:
                n = max(1, by_shm // max(1, info.gpu_count))
                p.notes.append(
                    f"/dev/shm is only {info.shm_free_mb:.0f} MB -> capped to "
                    f"{n} island(s) per GPU")
        by_host = max(1, int(info.avail_ram_gb * 1024 * 0.5
                             / (CUDA_HOST_MB * max(1, info.gpu_count))))
        n = max(1, min(n, by_host))
        p.islands_per_gpu = n
        p.gpu_islands = n * info.gpu_count
        usable = free - n * (GPU_RUNTIME_MB + dataset_mb)
        p.gpu_mem_budget_mb = int(max(128, usable / n))

        cc = min(float(g.compute_capability) for g in info.gpus
                 if g.compute_capability != "?") if info.gpus else 0.0
        p.use_tf32 = cc >= 8.0
        p.precision = "bf16" if cc >= 8.0 else "fp16"
        p.pin_memory = True
        p.notes.append(
            f"{p.islands_per_gpu} island(s) per GPU x {info.gpu_count} GPU(s) "
            f"= {p.gpu_islands} GPU islands ({free} MB free VRAM each)")
        if p.use_tf32:
            p.notes.append("Ampere+ detected -> TF32 matmul + bf16 autocast")

    # ---- dataset placement ------------------------------------------------- #
    if info.gpu_count:
        free = min((g.free_mb or g.memory_mb) for g in info.gpus)
        if dataset_mb * p.islands_per_gpu < free * 0.45:
            p.data_placement = "vram"
            p.notes.append(f"dataset ({dataset_mb:.0f} MB) pinned in VRAM - "
                           "batches never cross PCIe")
        elif dataset_mb < info.avail_ram_gb * 1024 * 0.35:
            p.data_placement = "ram"
        else:
            p.data_placement = "mmap"
    else:
        p.data_placement = ("ram" if dataset_mb < info.avail_ram_gb * 1024 * 0.35
                            else "mmap")
    if p.data_placement == "ram":
        p.notes.append(f"dataset ({dataset_mb:.0f} MB) held in system RAM")
    elif p.data_placement == "mmap":
        p.notes.append(f"dataset ({dataset_mb:.0f} MB) memory-mapped "
                       "(page cache shared between islands)")

    # ---- CPU islands: use whatever physical cores the GPUs don't need ----- #
    reserve = p.gpu_islands                      # ~1 core feeding each GPU island
    spare = max(0, info.cpu_physical - reserve - (1 if info.gpu_count else 0))
    # the orchestrator holds torch + (maybe) the dataset; forked islands then
    # cost only their marginal private heap on top of the shared pages
    parent_mb = CPU_RUNTIME_MB + (dataset_mb if p.data_placement == "ram" else 0)
    ram_for_cpu = info.avail_ram_gb * 1024 * 0.85 - parent_mb
    by_ram = int(max(0, ram_for_cpu // (CPU_FORK_MARGINAL_MB + MIN_ACT_MB)))

    if force_cpu_islands is not None:
        p.cpu_islands = max(0, force_cpu_islands)
    elif not allow_cpu_islands:
        p.cpu_islands = 0
    elif info.gpu_count == 0:
        p.cpu_islands = max(1, min(info.cpu_logical, by_ram))
    else:
        p.cpu_islands = max(0, min(spare, by_ram, 8))

    if p.cpu_islands:
        p.threads_per_cpu_island = max(
            1, (info.cpu_physical - reserve) // max(1, p.cpu_islands)) \
            if info.gpu_count else max(1, info.cpu_physical // p.cpu_islands)
        pool = (info.avail_ram_gb * 1024 * 0.85 - CPU_RUNTIME_MB
                - CPU_FORK_MARGINAL_MB * p.cpu_islands)
        if p.data_placement == "ram":
            pool -= dataset_mb
        p.cpu_mem_budget_mb = int(max(64, pool / p.cpu_islands))
        if info.gpu_count:
            p.notes.append(
                f"{p.cpu_islands} spare-core CPU island(s) alongside the GPUs "
                "(deadline-capped so they can never stall a generation)")

    p.threads_per_gpu_island = 1
    p.total_islands = max(1, p.gpu_islands + p.cpu_islands)
    return p


# --------------------------------------------------------------------------- #
# throughput calibration                                                       #
# --------------------------------------------------------------------------- #
def calibrate(devices: list[str], steps: int = 12) -> dict[str, float]:
    """Steps/second on a representative tiny workload, per device kind."""
    try:
        import torch
        import torch.nn as nn
    except ImportError:
        return {}
    out = {}
    for dev in dict.fromkeys(devices):
        try:
            d = torch.device(dev)
            m = nn.Sequential(nn.Flatten(), nn.Linear(64 * 57, 256), nn.GELU(),
                              nn.Linear(256, 9)).to(d)
            o = torch.optim.AdamW(m.parameters(), lr=1e-3)
            x = torch.randn(256, 64, 57, device=d)
            y = torch.randn(256, 9, device=d)
            for _ in range(3):                      # warm-up
                o.zero_grad(); nn.functional.mse_loss(m(x), y).backward(); o.step()
            if d.type == "cuda":
                torch.cuda.synchronize(d)
            t0 = time.time()
            for _ in range(steps):
                o.zero_grad(); nn.functional.mse_loss(m(x), y).backward(); o.step()
            if d.type == "cuda":
                torch.cuda.synchronize(d)
            out[dev] = round(steps / max(time.time() - t0, 1e-6), 1)
        except Exception:
            out[dev] = 0.0
    return out


# --------------------------------------------------------------------------- #
# main probe                                                                   #
# --------------------------------------------------------------------------- #
def detect(dataset_mb: float = 160.0, allow_cpu_islands: bool = True,
           force_cpu_islands: int | None = None,
           max_islands_per_gpu: int = 4) -> SystemInfo:
    info = SystemInfo()
    _cpu_details(info)

    try:
        import torch
        info.torch_version = torch.__version__
        info.cuda_available = torch.cuda.is_available()
        info.cuda_version = getattr(torch.version, "cuda", "") or ""
        info.mps_available = bool(getattr(getattr(torch.backends, "mps", None),
                                          "is_available", lambda: False)())
        if info.cuda_available:
            info.gpu_count = torch.cuda.device_count()
            for i in range(info.gpu_count):
                pr = torch.cuda.get_device_properties(i)
                try:
                    free, _ = torch.cuda.mem_get_info(i)
                    free_mb = free // (1024 ** 2)
                except Exception:
                    free_mb = pr.total_memory // (1024 ** 2)
                cc = f"{pr.major}.{pr.minor}"
                info.gpus.append(GPUInfo(
                    i, pr.name, pr.total_memory // (1024 ** 2), free_mb, cc,
                    getattr(pr, "multi_processor_count", 0),
                    pr.major >= 8, pr.major >= 8))
            try:
                info.nccl_available = torch.distributed.is_nccl_available()
            except Exception:
                pass
            info.p2p_matrix = _p2p_matrix(info.gpu_count)
    except ImportError:
        info.notes.append("PyTorch not installed - GPU probe via nvidia-smi only.")

    if info.gpu_count == 0:
        q = _run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free",
                  "--format=csv,noheader,nounits"])
        if q:
            for line in q.strip().splitlines():
                c = [x.strip() for x in line.split(",")]
                if len(c) >= 4:
                    info.gpus.append(GPUInfo(int(c[0]), c[1], int(c[2]), int(c[3])))
            info.gpu_count = len(info.gpus)

    drv = _run(["nvidia-smi", "--query-gpu=driver_version",
                "--format=csv,noheader"])
    if drv:
        info.driver_version = drv.strip().splitlines()[0]

    nv = detect_nvlink(info.gpu_count)
    info.nvlink_available = nv["available"]
    info.nvlink_active_links = nv["active_links"]
    info.nvlink_pairs = nv["pairs"]
    info.nvlink_fully_connected = nv["fully_connected"]
    info.nvlink_bandwidth_gbs = nv["bandwidth"]
    info.topology_matrix = nv["topology"]

    # ---- strategy ---------------------------------------------------------- #
    if info.gpu_count >= 2 and info.nvlink_available:
        info.strategy = "ddp-nvlink"
        info.world_size = info.gpu_count
        info.notes.append(
            f"NVLink: {info.nvlink_active_links} active links @ "
            f"{info.nvlink_bandwidth_gbs} GB/s, {len(info.nvlink_pairs)} peer pairs "
            "-> NCCL DDP with P2P.")
        if not info.nvlink_fully_connected and info.gpu_count > 2:
            info.notes.append("Partial NVLink mesh; some pairs fall back to PCIe.")
    elif info.gpu_count >= 2:
        info.strategy = "ddp-pcie"
        info.world_size = info.gpu_count
        info.notes.append("Multiple GPUs, no NVLink -> NCCL DDP over PCIe.")
    elif info.gpu_count == 1:
        info.strategy = "single-gpu-islands"
        info.world_size = 1
        info.notes.append("Single GPU -> concurrent islands share the device.")
    else:
        info.strategy = "cpu-islands"
        info.world_size = 1
        info.notes.append("No GPU -> parallel CPU islands.")

    if info.disk_free_gb < 1.0:
        info.blockers.append(
            f"Only {info.disk_free_gb:.2f} GB of disk free. Checkpoints, the "
            "feature cache and temp files cannot be written; workers will die "
            "with ENOSPC/ENOMEM. Free space, then retry.")
    if 0 < info.shm_total_mb < 256:
        info.blockers.append(
            f"/dev/shm is only {info.shm_total_mb:.0f} MB. PyTorch worker "
            "processes need far more and fail with 'OSError: [Errno 12] Cannot "
            "allocate memory'. Restart the container with --shm-size=16g "
            "(docker) or mount an emptyDir{medium: Memory} at /dev/shm (k8s).")

    info.plan = build_plan(info, dataset_mb, max_islands_per_gpu,
                           allow_cpu_islands, force_cpu_islands)
    if info.gpu_count >= 2:
        info.world_size = info.gpu_count
    return info


def apply_nvlink_env(info: SystemInfo) -> None:
    if info.strategy == "ddp-nvlink":
        os.environ["NCCL_P2P_DISABLE"] = "0"
        os.environ["NCCL_P2P_LEVEL"] = "NVL"
        os.environ["NCCL_NVLS_ENABLE"] = "1"
        os.environ.setdefault("NCCL_DEBUG", "WARN")
    elif info.strategy == "ddp-pcie":
        os.environ["NCCL_P2P_DISABLE"] = "1"
        os.environ.setdefault("NCCL_DEBUG", "WARN")


def apply_perf_env(info: SystemInfo) -> None:
    """Turn on every safe accelerator knob for the detected hardware."""
    try:
        import torch
    except ImportError:
        return
    p = info.plan
    if info.cuda_available:
        torch.backends.cudnn.benchmark = True
        if p.use_tf32:
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
            try:
                torch.set_float32_matmul_precision("high")
            except Exception:
                pass
    os.environ.setdefault("OMP_NUM_THREADS", str(max(1, p.threads_per_cpu_island)))
    os.environ.setdefault("MKL_NUM_THREADS", str(max(1, p.threads_per_cpu_island)))


def render(info: SystemInfo) -> str:
    p = info.plan
    L = ["=" * 74, "  COMPUTE DETECTION", "=" * 74]
    L.append(f"  CPU              : {info.cpu_model or 'unknown'}")
    L.append(f"       cores       : {info.cpu_physical} physical / "
             f"{info.cpu_logical} logical, {info.numa_nodes} NUMA node(s)")
    if info.cpu_flags:
        L.append(f"       ISA         : {' '.join(info.cpu_flags)}")
    L.append(f"  RAM              : {info.total_ram_gb} GB total, "
             f"{info.avail_ram_gb} GB available, {info.swap_gb} GB swap")
    L.append(f"  Disk free        : {info.disk_free_gb} GB"
             + ("   <-- FULL" if info.disk_free_gb < 1.0 else ""))
    L.append(f"  /dev/shm         : {info.shm_free_mb:.0f} MB free "
             f"/ {info.shm_total_mb:.0f} MB"
             + ("   <-- TOO SMALL" if 0 < info.shm_total_mb < 256 else ""))
    L.append(f"  PyTorch          : {info.torch_version}"
             + (f"  (CUDA {info.cuda_version})" if info.cuda_version else ""))
    if info.driver_version:
        L.append(f"  NVIDIA driver    : {info.driver_version}")
    L.append(f"  CUDA / NCCL      : {info.cuda_available} / {info.nccl_available}")
    L.append(f"  GPUs detected    : {info.gpu_count}")
    for g in info.gpus:
        L.append(f"      [{g.index}] {g.name}")
        L.append(f"           {g.free_mb}/{g.memory_mb} MB free, sm_{g.compute_capability}"
                 f", {g.multi_processors} SMs"
                 f"{', TF32+BF16' if g.supports_bf16 else ''}")
    L.append(f"  NVLink           : {'YES' if info.nvlink_available else 'NO'}")
    if info.nvlink_available:
        L.append(f"      links        : {info.nvlink_active_links} @ "
                 f"{info.nvlink_bandwidth_gbs} GB/s")
        L.append(f"      peer pairs   : {info.nvlink_pairs}")
        L.append(f"      full mesh    : {info.nvlink_fully_connected}")
    if info.p2p_matrix:
        L.append(f"      P2P matrix   : "
                 + " ".join("".join("1" if c else "0" for c in row)
                            for row in info.p2p_matrix))
    L.append("-" * 74)
    L.append(f"  STRATEGY         : {info.strategy}")
    L.append(f"  EXECUTION PLAN")
    L.append(f"      GPU islands  : {p.gpu_islands}"
             + (f"  ({p.islands_per_gpu} per GPU)" if p.gpu_islands else ""))
    L.append(f"      CPU islands  : {p.cpu_islands}"
             + (f"  ({p.threads_per_cpu_island} thread(s) each)"
                if p.cpu_islands else ""))
    L.append(f"      TOTAL        : {p.total_islands} concurrent training instances")
    L.append(f"      precision    : {p.precision}"
             + ("  (TF32 matmul on)" if p.use_tf32 else ""))
    L.append(f"      dataset      : {p.data_placement}  ({p.dataset_mb:.0f} MB)")
    if p.gpu_mem_budget_mb:
        L.append(f"      VRAM/island  : {p.gpu_mem_budget_mb} MB activation budget")
    if p.cpu_mem_budget_mb:
        L.append(f"      RAM/island   : {p.cpu_mem_budget_mb} MB activation budget")
    for n in info.notes + p.notes:
        L.append(f"    * {n}")
    if info.blockers:
        L.append("-" * 74)
        L.append("  BLOCKERS")
        for b in info.blockers:
            L.append(f"    !! {b}")
    L.append("=" * 74)
    if info.topology_matrix:
        L.append("\nnvidia-smi topo -m:\n" + info.topology_matrix)
    return "\n".join(L)

In [ ]:
SYS = detect()
apply_nvlink_env(SYS)
print(render(SYS))
(LOGS / "system_info.json").write_text(json.dumps(SYS.to_dict(), indent=2))

## 2 · Features and targets

57 strictly causal features — momentum, realised volatility, moving-average
distance, channel position, Bollinger, RSI/MACD/ATR, candle micro-structure,
volume z-scores, taker-buy flow imbalance, VWAP distance, return
autocorrelation and cyclical time-of-day / day-of-week.

Targets: forward log return `log(close[t+h]/close[t])` for
**1m, 5m, 15m, 30m, 1h, 2h, 4h, 8h, 12h**.

Splits are chronological with a **720-minute embargo** between them, so no
training row can peek at an outcome that overlaps validation or test.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# 1 minute -> 12 hours
HORIZONS: dict[str, int] = {
    "1m": 1, "5m": 5, "15m": 15, "30m": 30,
    "1h": 60, "2h": 120, "4h": 240, "8h": 480, "12h": 720,
}

DATA = ROOT / "data" / "processed" / "btc_1m.parquet"


# --------------------------------------------------------------------------- #
# indicators (all causal)                                                      #
# --------------------------------------------------------------------------- #
def _rsi(s: pd.Series, n: int) -> pd.Series:
    d = s.diff()
    up = d.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    return 100 - 100 / (1 + up / dn.replace(0, np.nan))


def _atr(df: pd.DataFrame, n: int) -> pd.Series:
    pc = df.close.shift(1)
    tr = pd.concat([df.high - df.low,
                    (df.high - pc).abs(),
                    (df.low - pc).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False).mean()


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    f = pd.DataFrame(index=df.index)
    c, lc = df.close, np.log(df.close)
    r1 = lc.diff()

    # --- momentum: log return over increasing lookbacks -------------------- #
    for w in (1, 2, 3, 5, 10, 15, 30, 60, 120, 240, 480, 720, 1440):
        f[f"ret_{w}"] = lc.diff(w)

    # --- realised volatility ------------------------------------------------ #
    for w in (5, 15, 60, 240, 1440):
        f[f"vol_{w}"] = r1.rolling(w).std()
    f["vol_ratio_60_1440"] = f["vol_60"] / f["vol_1440"].replace(0, np.nan)

    # --- price position vs moving averages ---------------------------------- #
    for w in (5, 15, 60, 240, 1440):
        ma = c.rolling(w).mean()
        f[f"ma_dist_{w}"] = (c - ma) / ma
    for w in (60, 240, 1440):
        lo, hi = df.low.rolling(w).min(), df.high.rolling(w).max()
        f[f"chan_pos_{w}"] = (c - lo) / (hi - lo).replace(0, np.nan)

    # --- Bollinger ----------------------------------------------------------- #
    for w in (60, 240):
        ma, sd = c.rolling(w).mean(), c.rolling(w).std()
        f[f"bb_{w}"] = (c - ma) / (2 * sd).replace(0, np.nan)

    # --- RSI / MACD / ATR ----------------------------------------------------- #
    for n in (14, 60, 240):
        f[f"rsi_{n}"] = _rsi(c, n) / 100 - 0.5
    ema12, ema26 = c.ewm(span=12 * 60).mean(), c.ewm(span=26 * 60).mean()
    macd = (ema12 - ema26) / c
    f["macd"] = macd
    f["macd_sig"] = macd.ewm(span=9 * 60).mean()
    f["macd_hist"] = f["macd"] - f["macd_sig"]
    for n in (60, 1440):
        f[f"atr_{n}"] = _atr(df, n) / c

    # --- candle micro-structure ------------------------------------------------ #
    rng = (df.high - df.low).replace(0, np.nan)
    f["body"] = (df.close - df.open) / c
    f["upper_wick"] = (df.high - df[["open", "close"]].max(axis=1)) / rng
    f["lower_wick"] = (df[["open", "close"]].min(axis=1) - df.low) / rng
    f["hl_range"] = rng / c

    # --- volume / flow ----------------------------------------------------------- #
    lv = np.log1p(df.volume)
    for w in (15, 60, 240, 1440):
        m, s = lv.rolling(w).mean(), lv.rolling(w).std()
        f[f"volz_{w}"] = (lv - m) / s.replace(0, np.nan)
    f["taker_ratio"] = (df.taker_buy_base / df.volume.replace(0, np.nan)) - 0.5
    for w in (15, 60, 240):
        f[f"taker_ratio_{w}"] = f["taker_ratio"].rolling(w).mean()
    f["trades_z"] = ((np.log1p(df.trades) - np.log1p(df.trades).rolling(1440).mean())
                     / np.log1p(df.trades).rolling(1440).std().replace(0, np.nan))
    vwap = (df.quote_volume.rolling(60).sum()
            / df.volume.rolling(60).sum().replace(0, np.nan))
    f["vwap_dist_60"] = (c - vwap) / c

    # --- autocorrelation / mean-reversion signal (vectorised) ------------------ #
    for w in (60, 240):
        f[f"ac_{w}"] = r1.rolling(w).corr(r1.shift(1))

    # --- seasonality (cyclical encodings) ---------------------------------------- #
    ts = df.ts.dt
    mins = ts.hour * 60 + ts.minute
    f["tod_sin"] = np.sin(2 * np.pi * mins / 1440)
    f["tod_cos"] = np.cos(2 * np.pi * mins / 1440)
    f["dow_sin"] = np.sin(2 * np.pi * ts.dayofweek / 7)
    f["dow_cos"] = np.cos(2 * np.pi * ts.dayofweek / 7)

    return f.replace([np.inf, -np.inf], np.nan)


def build_targets(df: pd.DataFrame) -> pd.DataFrame:
    lc = np.log(df.close)
    return pd.DataFrame(
        {f"y_{k}": lc.shift(-h) - lc for k, h in HORIZONS.items()},
        index=df.index)


# --------------------------------------------------------------------------- #
# dataset assembly with chronological split + embargo                          #
# --------------------------------------------------------------------------- #
def prepare(path: Path = DATA,
            train_frac: float = 0.70,
            val_frac: float = 0.15,
            subsample: int | None = None,
            verbose: bool = True) -> dict:
    df = pd.read_parquet(path)
    if subsample:
        df = df.iloc[-subsample:].reset_index(drop=True)

    X = build_features(df)
    Y = build_targets(df)

    warmup = 1440                      # rows whose features need history
    max_h = max(HORIZONS.values())     # rows whose targets look forward
    valid = np.zeros(len(df), bool)
    valid[warmup:len(df) - max_h] = True
    valid &= ~X.isna().any(axis=1).values
    valid &= ~Y.isna().any(axis=1).values
    idx = np.flatnonzero(valid)

    n = len(idx)
    i_tr, i_va = int(n * train_frac), int(n * (train_frac + val_frac))
    # embargo = longest horizon, so no training row can see a val/test outcome
    tr = idx[: max(0, i_tr - max_h)]
    va = idx[i_tr: max(i_tr, i_va - max_h)]
    te = idx[i_va:]

    Xv = X.values.astype(np.float32)
    Yv = Y.values.astype(np.float32)

    # standardise features on TRAIN ONLY (robust: median / IQR)
    med = np.nanmedian(Xv[tr], axis=0)
    q1, q3 = np.nanpercentile(Xv[tr], [25, 75], axis=0)
    scale = np.where((q3 - q1) > 1e-12, (q3 - q1) / 1.349, 1.0)
    Xn = np.clip((Xv - med) / scale, -8, 8)
    Xn = np.nan_to_num(Xn, nan=0.0).astype(np.float32)

    # scale targets to ~unit variance per horizon (train stats) so the loss
    # is not dominated by the 12h head
    ystd = Yv[tr].std(axis=0)
    ystd = np.where(ystd > 1e-12, ystd, 1.0).astype(np.float32)

    out = {
        "X": Xn, "Y": Yv, "y_std": ystd,
        "train_idx": tr, "val_idx": va, "test_idx": te,
        "feature_names": list(X.columns),
        "horizon_names": [f"y_{k}" for k in HORIZONS],
        "horizon_steps": list(HORIZONS.values()),
        "close": df.close.values.astype(np.float32),
        "ts": df.ts.values,
        "scaler": {"median": med, "scale": scale},
    }
    if verbose:
        print(f"features   : {Xn.shape[1]}")
        print(f"usable rows: {n:,}")
        print(f"  train    : {len(tr):,}  {df.ts.iloc[tr[0]].date()} -> {df.ts.iloc[tr[-1]].date()}")
        print(f"  val      : {len(va):,}  {df.ts.iloc[va[0]].date()} -> {df.ts.iloc[va[-1]].date()}")
        print(f"  test     : {len(te):,}  {df.ts.iloc[te[0]].date()} -> {df.ts.iloc[te[-1]].date()}")
        print(f"  embargo  : {max_h} min between splits")
    return out

In [ ]:
DATASET = prepare()          # ~10 s on the full year
print("\nfeature matrix:", DATASET["X"].shape, " targets:", DATASET["Y"].shape)
pd.DataFrame({
    "horizon": DATASET["horizon_names"],
    "minutes": DATASET["horizon_steps"],
    "target std (log-ret)": DATASET["y_std"],
}).set_index("horizon")

## 3 · Model zoo

Four backbones the genetic algorithm can choose between. Each maps a window
`(batch, lookback, n_features)` to `(batch, 9)` standardised log returns.

In [ ]:
import warnings

import torch
import torch.nn as nn

ACTS = {"relu": nn.ReLU, "gelu": nn.GELU, "silu": nn.SiLU, "tanh": nn.Tanh}


class MLP(nn.Module):
    def __init__(self, n_feat, lookback, n_out, hidden, depth, dropout, act):
        super().__init__()
        A = ACTS[act]
        layers, d = [nn.Flatten()], n_feat * lookback
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.LayerNorm(hidden), A(), nn.Dropout(dropout)]
            d = hidden
        layers.append(nn.Linear(d, n_out))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class GRUNet(nn.Module):
    def __init__(self, n_feat, lookback, n_out, hidden, depth, dropout, act):
        super().__init__()
        self.gru = nn.GRU(n_feat, hidden, num_layers=depth, batch_first=True,
                          dropout=dropout if depth > 1 else 0.0)
        self.head = nn.Sequential(nn.LayerNorm(hidden), ACTS[act](),
                                  nn.Dropout(dropout), nn.Linear(hidden, n_out))

    def forward(self, x):
        o, _ = self.gru(x)
        return self.head(o[:, -1])


class _TCNBlock(nn.Module):
    def __init__(self, cin, cout, k, dil, dropout, act):
        super().__init__()
        self.pad = (k - 1) * dil                      # causal padding
        self.conv = nn.Conv1d(cin, cout, k, dilation=dil)
        self.norm = nn.GroupNorm(1, cout)
        self.act = ACTS[act]()
        self.drop = nn.Dropout(dropout)
        self.res = nn.Conv1d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x):
        y = self.conv(nn.functional.pad(x, (self.pad, 0)))
        return self.drop(self.act(self.norm(y))) + self.res(x)


class TCN(nn.Module):
    def __init__(self, n_feat, lookback, n_out, hidden, depth, dropout, act):
        super().__init__()
        self.blocks = nn.ModuleList([
            _TCNBlock(n_feat if i == 0 else hidden, hidden, 3, 2 ** i, dropout, act)
            for i in range(depth)])
        self.head = nn.Sequential(nn.Flatten(), nn.LayerNorm(hidden),
                                  nn.Dropout(dropout), nn.Linear(hidden, n_out))

    def forward(self, x):
        h = x.transpose(1, 2)
        for b in self.blocks:
            h = b(h)
        return self.head(h[:, :, -1])


class AttnNet(nn.Module):
    """Small causal transformer encoder over the lookback window."""

    def __init__(self, n_feat, lookback, n_out, hidden, depth, dropout, act):
        super().__init__()
        heads = max(1, min(8, hidden // 32))
        while hidden % heads:
            heads -= 1
        self.inp = nn.Linear(n_feat, hidden)
        self.pos = nn.Parameter(torch.randn(1, lookback, hidden) * 0.02)
        layer = nn.TransformerEncoderLayer(
            hidden, heads, dim_feedforward=hidden * 2, dropout=dropout,
            activation="gelu", batch_first=True, norm_first=True)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', UserWarning)
            self.enc = nn.TransformerEncoder(layer, depth)
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, n_out))
        self.register_buffer(
            "mask", torch.triu(torch.ones(lookback, lookback, dtype=torch.bool), 1))

    def forward(self, x):
        h = self.enc(self.inp(x) + self.pos, mask=self.mask)
        return self.head(h[:, -1])


BACKBONES = {"mlp": MLP, "gru": GRUNet, "tcn": TCN, "attn": AttnNet}


def build_model(genome: dict, n_feat: int, n_out: int) -> nn.Module:
    cls = BACKBONES[genome["backbone"]]
    return cls(n_feat, genome["lookback"], n_out,
               genome["hidden"], genome["depth"],
               genome["dropout"], genome["act"])


def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 4 · Genetic operators

A genome encodes both **architecture** and **optimisation** hyper-parameters.
Reproduction = elitism + tournament selection + BLX-α crossover + per-gene
mutation + a trickle of random immigrants to keep diversity up.

In [ ]:
import copy
import hashlib
import json
import math
import random

# gene name -> list of allowed values (categorical) or (lo, hi, "log"|"lin")
GENE_SPACE = {
    "backbone":    ["mlp", "gru", "tcn", "attn"],
    "lookback":    [16, 32, 64, 96, 128],
    "hidden":      [32, 48, 64, 96, 128, 192, 256],
    "depth":       [1, 2, 3, 4],
    "dropout":     (0.0, 0.45, "lin"),
    "lr":          (3e-5, 8e-3, "log"),
    "weight_decay": (1e-7, 1e-2, "log"),
    "batch_size":  [128, 256, 512, 1024],
    "act":         ["relu", "gelu", "silu", "tanh"],
    "huber_delta": (0.25, 3.0, "log"),
    "feat_drop":   (0.0, 0.30, "lin"),
    "grad_clip":   (0.2, 5.0, "log"),
    # per-horizon loss weighting exponent: <0 favours short horizons
    "horizon_w":   (-0.6, 0.6, "lin"),
}

# --- conservative defaults ------------------------------------------------- #
MUTATE_RATE = 0.12       # fraction of genes touched per child
MUTATE_SIGMA = 0.15      # jitter size for continuous genes
ELITE_FRAC = 0.30        # survivors copied verbatim
IMMIGRANT_FRAC = 0.04    # fresh random blood


def random_genome(rng: random.Random) -> dict:
    g = {}
    for k, sp in GENE_SPACE.items():
        if isinstance(sp, list):
            g[k] = rng.choice(sp)
        else:
            lo, hi, mode = sp
            g[k] = (math.exp(rng.uniform(math.log(lo), math.log(hi)))
                    if mode == "log" else rng.uniform(lo, hi))
    return g


def mutate(genome: dict, rng: random.Random, rate: float = MUTATE_RATE,
           sigma: float = MUTATE_SIGMA) -> dict:
    """Per-gene mutation: categorical resample or log/linear Gaussian jitter."""
    g = copy.deepcopy(genome)
    for k, sp in GENE_SPACE.items():
        if rng.random() > rate:
            continue
        if isinstance(sp, list):
            if k in ("lookback", "hidden", "depth"):     # ordered -> step neighbour
                i = sp.index(g[k])
                g[k] = sp[max(0, min(len(sp) - 1, i + rng.choice([-1, 1])))]
            else:
                g[k] = rng.choice(sp)
        else:
            lo, hi, mode = sp
            v = (math.exp(math.log(g[k]) + rng.gauss(0, sigma)) if mode == "log"
                 else g[k] + rng.gauss(0, sigma * (hi - lo)))
            g[k] = max(lo, min(hi, v))
    return g


def crossover(a: dict, b: dict, rng: random.Random) -> dict:
    """Uniform crossover; BLX-alpha blend for continuous genes."""
    c = {}
    for k, sp in GENE_SPACE.items():
        if isinstance(sp, list):
            c[k] = a[k] if rng.random() < 0.5 else b[k]
        else:
            lo, hi, _ = sp
            w = rng.uniform(-0.1, 1.1)
            c[k] = max(lo, min(hi, a[k] * w + b[k] * (1 - w)))
    return c


def tournament(pop: list[dict], rng: random.Random, k: int = 3) -> dict:
    """pop entries are {'genome':..., 'fitness': float} - LOWER is better."""
    cands = rng.sample(pop, min(k, len(pop)))
    return min(cands, key=lambda d: d["fitness"])["genome"]


def next_generation(scored: list[dict], pop_size: int, rng: random.Random,
                    elite_frac: float = ELITE_FRAC,
                    mutate_rate: float = MUTATE_RATE,
                    immigrant_frac: float = IMMIGRANT_FRAC,
                    mutate_sigma: float = MUTATE_SIGMA) -> list[dict]:
    """scored: [{'genome':..., 'fitness':...}]. Returns the next population."""
    scored = sorted(scored, key=lambda d: d["fitness"])
    n_elite = max(1, int(round(pop_size * elite_frac)))
    n_immig = int(pop_size * immigrant_frac)

    children = [copy.deepcopy(s["genome"]) for s in scored[:n_elite]]   # elitism
    while len(children) < pop_size - n_immig:
        p1, p2 = tournament(scored, rng), tournament(scored, rng)
        children.append(mutate(crossover(p1, p2, rng), rng,
                               mutate_rate, mutate_sigma))
    while len(children) < pop_size:
        children.append(random_genome(rng))
    return children


def genome_key(g: dict) -> str:
    """Stable short signature, used for caching evaluations across resumes."""
    s = json.dumps({k: (round(v, 6) if isinstance(v, float) else v)
                    for k, v in sorted(g.items())}, sort_keys=True)
    return hashlib.sha1(s.encode()).hexdigest()[:12]


def cost_proxy(g: dict, n_feat: int = 57) -> float:
    """Relative training cost, used to hand cheap genomes to CPU islands."""
    B, L, H, D = g["batch_size"], g["lookback"], g["hidden"], g["depth"]
    if g["backbone"] == "mlp":
        return B * (L * n_feat * H + H * H * max(0, D - 1))
    if g["backbone"] == "attn":
        return B * L * (H * H * D * 4 + L * H * D)
    return B * L * H * H * D            # gru / tcn

In [ ]:
_demo_rng = random.Random(0)
pd.DataFrame([random_genome(_demo_rng) for _ in range(5)])

## 5 · Evolutionary trainer

Each individual is trained for `inner_steps` gradient steps, then scored on
validation by its **skill** = MSE ÷ MSE-of-predicting-zero, averaged over the
nine horizons (plus a small parsimony penalty). Lower is better; **< 1.0 beats
the naive baseline**.

Robustness built in:

* **Memory guard** — predicts each genome's activation footprint and shrinks
  its batch size; an over-budget genome is scored badly instead of OOM-killing
  the run.
* **Crash isolation** — a worker that dies is retried serially; one bad
  individual can never take down the evolution.
* **Resumability** — `checkpoints/state.json` is written atomically every
  generation with the population, RNG state and cumulative elapsed time.
  Re-running resumes exactly where it stopped, and SIGINT/SIGTERM checkpoints
  before exiting.
* **Wall-clock budget** — `--max-hours 5` stops cleanly and saves the model.

In [ ]:
import argparse
import json
import os
import pickle
import random
import signal
import sys
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from contextlib import ExitStack
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn


ROOT = ROOT
CACHE = ROOT / "data" / "processed" / "cache"
CKPT = Path(os.environ.get("BTC_CKPT_DIR", ROOT / "checkpoints"))
LOGS = ROOT / "logs"
_STOP = {"flag": False}
_DATA: dict | None = None          # process-global; forked islands inherit it


# --------------------------------------------------------------------------- #
# data cache                                                                   #
# --------------------------------------------------------------------------- #
def build_cache(subsample: int | None = None, force: bool = False) -> dict:
    CACHE.mkdir(parents=True, exist_ok=True)
    meta_p = CACHE / "meta.json"
    if meta_p.exists() and not force:
        cached = json.loads(meta_p.read_text())
        # A cache built with a different --subsample silently changes the size
        # of the test set, which makes every reported metric incomparable.
        # Rebuild instead of quietly reusing it.
        if cached.get("subsample") == subsample:
            return cached
        print(f"  [cache] built with subsample={cached.get('subsample')} but "
              f"subsample={subsample} was requested -> rebuilding", flush=True)

    d = prepare(subsample=subsample)
    np.save(CACHE / "X.npy", d["X"])
    np.save(CACHE / "Y.npy", d["Y"])
    for k in ("train_idx", "val_idx", "test_idx"):
        np.save(CACHE / f"{k}.npy", d[k].astype(np.int64))
    np.save(CACHE / "y_std.npy", d["y_std"])
    np.save(CACHE / "close.npy", d["close"])
    with open(CACHE / "scaler.pkl", "wb") as f:
        pickle.dump(d["scaler"], f)
    meta = {
        "n_features": int(d["X"].shape[1]),
        "n_horizons": int(d["Y"].shape[1]),
        "feature_names": d["feature_names"],
        "horizon_names": d["horizon_names"],
        "horizon_steps": d["horizon_steps"],
        "n_rows": int(d["X"].shape[0]),
        "n_train": int(len(d["train_idx"])),
        "n_val": int(len(d["val_idx"])),
        "n_test": int(len(d["test_idx"])),
        "subsample": subsample,
        "bytes": int(d["X"].nbytes + d["Y"].nbytes),
    }
    meta_p.write_text(json.dumps(meta, indent=2))
    return meta


def dataset_mb() -> float:
    try:
        m = json.loads((CACHE / "meta.json").read_text())
        if m.get("bytes"):
            return m["bytes"] / 1e6
        return (m["n_rows"] * (m["n_features"] + m["n_horizons"]) * 4) / 1e6
    except Exception:
        return 160.0


def load_cache(placement: str = "mmap", reuse: bool = True) -> dict:
    """placement: 'ram' loads into memory, anything else memory-maps."""
    global _DATA
    if reuse and _DATA is not None:
        return _DATA
    # The derived cache is regenerable and therefore not committed. On a fresh
    # clone it simply will not be there - build it rather than exploding.
    if not (CACHE / "meta.json").exists():
        print("  [cache] not present - building it now (one-off, ~20 s)",
              flush=True)
        build_cache()
    m = None if placement == "ram" else "r"
    d = {
        "X": np.load(CACHE / "X.npy", mmap_mode=m),
        "Y": np.load(CACHE / "Y.npy", mmap_mode=m),
        "train_idx": np.load(CACHE / "train_idx.npy"),
        "val_idx": np.load(CACHE / "val_idx.npy"),
        "test_idx": np.load(CACHE / "test_idx.npy"),
        "y_std": np.load(CACHE / "y_std.npy"),
        "meta": json.loads((CACHE / "meta.json").read_text()),
    }
    if reuse:
        _DATA = d
    return d


class WindowSource:
    """Serves (B, L, F) windows, from GPU-resident tensors when possible."""

    def __init__(self, X, Y, device: torch.device, on_device: bool):
        self.device, self.on_device = device, on_device
        if on_device:
            self.X = torch.as_tensor(np.ascontiguousarray(X)).to(device)
            self.Y = torch.as_tensor(np.ascontiguousarray(Y)).to(device)
        else:
            self.X, self.Y = X, Y

    def batch(self, idx: np.ndarray, offsets, ystd):
        if self.on_device:
            i = torch.as_tensor(idx, device=self.device, dtype=torch.long)
            xb = self.X[i[:, None] + offsets]
            yb = self.Y[i] / ystd
            return xb, yb
        win = idx[:, None] + offsets
        xb = torch.from_numpy(np.ascontiguousarray(self.X[win])).to(
            self.device, non_blocking=True)
        yb = torch.from_numpy(np.ascontiguousarray(self.Y[idx])).to(
            self.device, non_blocking=True)
        return xb, yb / ystd


# --------------------------------------------------------------------------- #
# memory guard                                                                 #
# --------------------------------------------------------------------------- #
def estimate_peak_mb(g: dict, n_feat: int) -> float:
    B, L, H, D = g["batch_size"], g["lookback"], g["hidden"], g["depth"]
    inp = B * L * n_feat * 4
    bb = g["backbone"]
    if bb == "mlp":
        act = B * H * 4 * D * 4 + B * L * n_feat * 4 * 2
    elif bb in ("gru", "tcn"):
        act = B * L * H * 4 * D * 6
    else:
        heads = max(1, min(8, H // 32))
        act = B * L * H * 4 * D * 8 + B * heads * L * L * 4 * D * 3
    return (inp + act) * 3.0 / 1e6


LOOKBACKS = [16, 32, 64, 96, 128]


def fit_to_memory(g: dict, n_feat: int, budget_mb: float) -> tuple[dict, bool]:
    """Adapt the genome to the island it landed on: batch first, then window."""
    g = dict(g)
    while estimate_peak_mb(g, n_feat) > budget_mb and g["batch_size"] > 32:
        g["batch_size"] //= 2
    while estimate_peak_mb(g, n_feat) > budget_mb and g["lookback"] > LOOKBACKS[0]:
        smaller = [l for l in LOOKBACKS if l < g["lookback"]]
        g["lookback"] = smaller[-1]
    return g, estimate_peak_mb(g, n_feat) <= budget_mb


def _limit_address_space(mb: int) -> None:
    try:
        import resource
        soft, hard = resource.getrlimit(resource.RLIMIT_AS)
        cap = int(mb * 1024 * 1024)
        if hard != resource.RLIM_INFINITY:
            cap = min(cap, hard)
        resource.setrlimit(resource.RLIMIT_AS, (cap, hard))
    except Exception:
        pass


def _autocast(device: torch.device, precision: str):
    if device.type == "cuda" and precision in ("bf16", "fp16"):
        dt = torch.bfloat16 if precision == "bf16" else torch.float16
        return torch.amp.autocast("cuda", dtype=dt), precision == "fp16"
    import contextlib
    return contextlib.nullcontext(), False


# --------------------------------------------------------------------------- #
# distributed (torchrun) support                                               #
# --------------------------------------------------------------------------- #
def dist_init():
    import torch.distributed as dist
    if "RANK" not in os.environ or "WORLD_SIZE" not in os.environ:
        return False, 0, 1, 0
    local = int(os.environ.get("LOCAL_RANK", 0))
    if torch.cuda.is_available():
        torch.cuda.set_device(local % torch.cuda.device_count())
        backend = "nccl"
    else:
        backend = "gloo"
    dist.init_process_group(backend=backend)
    return True, dist.get_rank(), dist.get_world_size(), local


def ddp_evaluate(task: dict, rank: int, world: int) -> dict:
    """All ranks cooperate on ONE individual; gradients all-reduce over NVLink."""
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP

    g = task["genome"]
    device = torch.device(task["device"])
    D = load_cache(task.get("placement", "mmap"))
    tr, va = D["train_idx"], D["val_idx"]
    n_feat, n_out = D["meta"]["n_features"], D["meta"]["n_horizons"]
    ystd = torch.from_numpy(D["y_std"]).to(device)

    g, _ = fit_to_memory(g, n_feat, task.get("mem_budget_mb", 400))
    L = g["lookback"]
    tr, va = tr[tr >= L - 1], va[va >= L - 1]
    src = WindowSource(D["X"], D["Y"], device, task.get("on_device", False))
    offsets = torch.arange(-L + 1, 1, device=device) if src.on_device \
        else np.arange(-L + 1, 1, dtype=np.int64)

    torch.manual_seed(task["seed"])
    base = build_model(g, n_feat, n_out).to(device)
    n_par = count_params(base)
    model = DDP(base, device_ids=[device.index] if device.type == "cuda" else None)
    opt = torch.optim.AdamW(model.parameters(), lr=g["lr"],
                            weight_decay=g["weight_decay"])
    steps = task["inner_steps"]
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=g["lr"],
                                                total_steps=steps, pct_start=0.25)
    hs = np.asarray(D["meta"]["horizon_steps"], dtype=np.float64)
    w = hs ** g["horizon_w"]
    w = torch.tensor(w / w.mean(), dtype=torch.float32, device=device)
    lossfn = nn.HuberLoss(delta=g["huber_delta"], reduction="none")
    ac, need_scaler = _autocast(device, task.get("precision", "fp32"))
    scaler = torch.amp.GradScaler("cuda", enabled=need_scaler)
    shard = max(32, g["batch_size"] // world)
    rng = np.random.default_rng(task["seed"] + rank)

    t0 = time.time()
    model.train()
    for _ in range(steps):
        bidx = tr[rng.integers(0, len(tr), size=shard)]
        xb, yb = src.batch(bidx, offsets, ystd)
        with ac:
            loss = (lossfn(model(xb), yb) * w).mean()
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), g["grad_clip"])
        scaler.step(opt); scaler.update(); sched.step()

    model.eval()
    part = va[rank::world]
    se = torch.zeros(n_out, device=device)
    var = torch.zeros(n_out, device=device)
    hit = torch.zeros(n_out, device=device)
    cnt = torch.zeros(1, device=device)
    with torch.no_grad():
        vb = max(256, min(8192, g["batch_size"] * 4))
        for i in range(0, len(part), vb):
            b = part[i:i + vb]
            xb, yb = src.batch(b, offsets, ystd)
            with ac:
                p = base(xb)
            p = p.float()
            se += ((p - yb) ** 2).sum(0)
            var += (yb ** 2).sum(0)
            hit += ((p.sign() == yb.sign()) & (yb != 0)).sum(0)
            cnt += len(b)
    for t in (se, var, hit, cnt):
        dist.all_reduce(t, op=dist.ReduceOp.SUM)
    skill = (se / var.clamp_min(1e-12)).cpu().numpy()
    dacc = (hit / cnt.clamp_min(1)).cpu().numpy()

    state_path = None
    if rank == 0 and task.get("save_state"):
        state_path = str(task["save_state"])
        torch.save(base.state_dict(), state_path + ".tmp")
        os.replace(state_path + ".tmp", state_path)
    dist.barrier()

    return {"genome": g, "key": genome_key(g),
            "fitness": float(np.mean(skill)) + task.get("parsimony", 0) * n_par / 1e6,
            "skill_per_h": skill.tolist(), "mean_skill": float(np.mean(skill)),
            "dir_acc": dacc.tolist(), "mean_dir_acc": float(np.mean(dacc)),
            "params": int(n_par), "seconds": round(time.time() - t0, 1),
            "state_path": state_path, "steps_done": steps, "truncated": False,
            "device": task["device"]}


# --------------------------------------------------------------------------- #
# fitness evaluation of one genome  (= one training instance)                  #
# --------------------------------------------------------------------------- #
def evaluate_genome(task: dict) -> dict:
    """Crash-safe wrapper: a bad genome scores badly, it never kills the run."""
    try:
        return _evaluate_inner(task)
    except MemoryError:
        err = "MemoryError"
    except RuntimeError as e:
        err = f"RuntimeError: {str(e)[:160]}"
    except Exception as e:                                  # noqa: BLE001
        err = f"{type(e).__name__}: {str(e)[:160]}"
    return _dead(task, err)


def _dead(task: dict, why: str) -> dict:
    g = task["genome"]
    return {"genome": g, "key": genome_key(g), "fitness": 1e6, "skill_per_h": [],
            "mean_skill": float("nan"), "dir_acc": [], "mean_dir_acc": float("nan"),
            "params": 0, "seconds": 0.0, "state_path": None, "steps_done": 0,
            "truncated": True, "error": why, "device": task.get("device", "?")}


def _evaluate_inner(task: dict) -> dict:
    g = task["genome"]
    device = torch.device(task["device"])
    torch.set_num_threads(task.get("threads", 1))
    if device.type == "cpu" and task.get("rlimit_mb"):
        # never cap address space when CUDA is in the picture: the driver
        # reserves tens of GB of *virtual* memory and the cap turns that into
        # a spurious ENOMEM
        _limit_address_space(task["rlimit_mb"])
    seed = task["seed"]
    torch.manual_seed(seed)
    np.random.seed(seed % (2 ** 31))

    D = load_cache(task.get("placement", "mmap"))
    tr, va = D["train_idx"], D["val_idx"]
    n_feat, n_out = D["meta"]["n_features"], D["meta"]["n_horizons"]
    ystd = torch.from_numpy(D["y_std"]).to(device)

    g, fits = fit_to_memory(g, n_feat, task.get("mem_budget_mb", 400))
    if not fits:
        return _dead({**task, "genome": g},
                     f"too large ({estimate_peak_mb(g, n_feat):.0f}MB)")

    L = g["lookback"]
    tr, va = tr[tr >= L - 1], va[va >= L - 1]
    on_dev = bool(task.get("on_device", False))
    src = WindowSource(D["X"], D["Y"], device, on_dev)
    offsets = torch.arange(-L + 1, 1, device=device) if on_dev \
        else np.arange(-L + 1, 1, dtype=np.int64)

    model = build_model(g, n_feat, n_out).to(device)
    n_par = count_params(model)
    opt = torch.optim.AdamW(model.parameters(), lr=g["lr"],
                            weight_decay=g["weight_decay"])
    steps = task["inner_steps"]
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=g["lr"], total_steps=steps, pct_start=0.25)

    hs = np.asarray(D["meta"]["horizon_steps"], dtype=np.float64)
    w = hs ** g["horizon_w"]
    w = torch.tensor(w / w.mean(), dtype=torch.float32, device=device)

    lossfn = nn.HuberLoss(delta=g["huber_delta"], reduction="none")
    bs = g["batch_size"]
    rng = np.random.default_rng(seed)
    ac, need_scaler = _autocast(device, task.get("precision", "fp32"))
    scaler = torch.amp.GradScaler("cuda", enabled=need_scaler)

    if task.get("init_state") and Path(task["init_state"]).exists():
        try:
            sd = torch.load(task["init_state"], map_location=device,
                            weights_only=True)
            model.load_state_dict(sd, strict=True)
        except Exception:
            pass

    t0 = time.time()
    model.train()
    step = -1
    for step in range(steps):
        bidx = tr[rng.integers(0, len(tr), size=min(bs, len(tr)))]
        xb, yb = src.batch(bidx, offsets, ystd)
        if g["feat_drop"] > 0:
            mask = (torch.rand(1, 1, n_feat, device=device) > g["feat_drop"])
            xb = xb * mask.to(xb.dtype)
        with ac:
            loss = (lossfn(model(xb), yb) * w).mean()
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), g["grad_clip"])
        scaler.step(opt); scaler.update(); sched.step()
        if task.get("deadline") and (step & 31) == 0 and time.time() > task["deadline"]:
            break
    steps_done = step + 1
    truncated = steps_done < steps

    model.eval()
    se = np.zeros(n_out); var = np.zeros(n_out); hit = np.zeros(n_out); n = 0
    with torch.no_grad():
        vb = max(256, min(8192, bs * 4))
        for i in range(0, len(va), vb):
            bidx = va[i:i + vb]
            xb, yb = src.batch(bidx, offsets, ystd)
            with ac:
                p = model(xb)
            p = p.float()
            se += ((p - yb) ** 2).sum(0).cpu().numpy()
            var += (yb ** 2).sum(0).cpu().numpy()
            hit += ((p.sign() == yb.sign()) & (yb != 0)).sum(0).cpu().numpy()
            n += len(bidx)

    mse, base = se / max(n, 1), var / max(n, 1)
    skill = mse / np.maximum(base, 1e-12)
    dacc = hit / max(n, 1)

    fitness = float(np.mean(skill)) + task.get("parsimony", 0.0) * n_par / 1e6
    if not np.isfinite(fitness):
        fitness = 1e6

    state_path = None
    if task.get("save_state"):
        state_path = str(Path(task["save_state"]))
        torch.save(model.state_dict(), state_path + ".tmp")
        os.replace(state_path + ".tmp", state_path)

    return {
        "genome": g, "key": genome_key(g), "fitness": fitness,
        "skill_per_h": skill.tolist(), "mean_skill": float(np.mean(skill)),
        "dir_acc": dacc.tolist(), "mean_dir_acc": float(np.mean(dacc)),
        "params": int(n_par), "seconds": round(time.time() - t0, 1),
        "state_path": state_path, "steps_done": steps_done,
        "truncated": truncated, "device": task["device"],
    }


# --------------------------------------------------------------------------- #
# heterogeneous population evaluation                                          #
# --------------------------------------------------------------------------- #
def evaluate_population(tasks: list[dict], n_gpu: int, n_cpu: int,
                        ctx_gpu, ctx_cpu) -> list[dict]:
    """GPU islands and CPU islands chew through the population concurrently.

    A worker killed by the OS is retried serially, so one bad individual can
    never take down the evolution run.
    """
    gpu_tasks = [(i, t) for i, t in enumerate(tasks) if t["device"] != "cpu"]
    cpu_tasks = [(i, t) for i, t in enumerate(tasks) if t["device"] == "cpu"]

    if (n_gpu + n_cpu) <= 1 or len(tasks) == 1:
        return [evaluate_genome(t) for t in tasks]

    results: dict[int, dict] = {}
    try:
        with ExitStack() as stack:
            futs = {}
            if gpu_tasks:
                gex = stack.enter_context(ProcessPoolExecutor(
                    max_workers=max(1, min(n_gpu, len(gpu_tasks))),
                    mp_context=ctx_gpu))
                for i, t in gpu_tasks:
                    futs[gex.submit(evaluate_genome, t)] = i
            if cpu_tasks:
                cex = stack.enter_context(ProcessPoolExecutor(
                    max_workers=max(1, min(n_cpu, len(cpu_tasks))),
                    mp_context=ctx_cpu))
                for i, t in cpu_tasks:
                    futs[cex.submit(evaluate_genome, t)] = i
            for fut in as_completed(futs):
                i = futs[fut]
                try:
                    results[i] = fut.result()
                except Exception as e:                       # noqa: BLE001
                    results[i] = _dead(tasks[i], f"worker died: {type(e).__name__}")
    except Exception as e:                                   # noqa: BLE001
        print(f"  [pool broken: {type(e).__name__}: {str(e)[:200]}] "
              "retrying survivors serially", flush=True)

    out = []
    for i, t in enumerate(tasks):
        r = results.get(i)
        if r is not None and "worker died" not in str(r.get("error", "")):
            out.append(r)
        else:
            out.append(evaluate_genome(t))
    return out


# --------------------------------------------------------------------------- #
# checkpointing                                                                #
# --------------------------------------------------------------------------- #
def save_state(state: dict) -> None:
    CKPT.mkdir(parents=True, exist_ok=True)
    tmp = CKPT / "state.json.tmp"
    tmp.write_text(json.dumps(state, indent=2, default=str))
    os.replace(tmp, CKPT / "state.json")


def load_state() -> dict | None:
    p = CKPT / "state.json"
    if not p.exists():
        return None
    try:
        return json.loads(p.read_text())
    except json.JSONDecodeError:
        return None


def _handler(signum, frame):
    _STOP["flag"] = True
    print(f"\n[signal {signum}] finishing current generation, then checkpointing...",
          flush=True)


# --------------------------------------------------------------------------- #
# main evolution loop                                                          #
# --------------------------------------------------------------------------- #
def run(args) -> None:
    signal.signal(signal.SIGINT, _handler)
    signal.signal(signal.SIGTERM, _handler)
    for d in (CKPT, CKPT / "elites", LOGS):
        d.mkdir(parents=True, exist_ok=True)

    # cache must exist before the planner can size anything
    pre = (CACHE / "meta.json").exists() and not args.rebuild_cache
    if not pre:
        build_cache(subsample=args.subsample, force=args.rebuild_cache)

    info = detect(dataset_mb=dataset_mb(),
                  allow_cpu_islands=not args.no_cpu_islands,
                  force_cpu_islands=(args.cpu_islands
                                     if args.cpu_islands >= 0 else None),
                  max_islands_per_gpu=args.islands_per_gpu)
    apply_nvlink_env(info)
    apply_perf_env(info)
    IS_DIST, RANK, WORLD, LOCAL = dist_init()
    main_proc = RANK == 0
    plan = info.plan

    def log(*a, **k):
        if main_proc:
            print(*a, **k, flush=True)

    if IS_DIST:
        import torch.distributed as dist
        log(f"[distributed] backend={dist.get_backend()} world_size={WORLD} "
            f"mode={args.parallel}")
    log(render(info))

    if info.blockers and not args.ignore_blockers:
        log("\n" + "!" * 70)
        log("REFUSING TO START - the environment cannot support a training run:")
        for b in info.blockers:
            log(f"  * {b}")
        log("")
        log("Run  python3 scripts/diagnose.py  for a full breakdown.")
        log("Override with --ignore-blockers if you are certain.")
        log("!" * 70)
        raise SystemExit(3)

    meta = build_cache(subsample=args.subsample, force=args.rebuild_cache)
    if main_proc:
        (LOGS / "system_info.json").write_text(json.dumps(info.to_dict(), indent=2))
    if IS_DIST:
        import torch.distributed as dist
        dist.barrier()

    log(f"\ndata: {meta['n_rows']:,} rows | {meta['n_features']} features | "
        f"{meta['n_horizons']} horizons | train {meta['n_train']:,} "
        f"val {meta['n_val']:,} test {meta['n_test']:,}\n")

    # ---- island layout ----------------------------------------------------- #
    if IS_DIST:
        # this rank owns one GPU; pack several islands on it and claim its
        # share of the spare CPU cores as extra islands
        dev = f"cuda:{LOCAL % info.gpu_count}" if info.gpu_count else "cpu"
        n_gpu_isl = max(1, plan.islands_per_gpu) if info.gpu_count else 0
        n_cpu_isl = plan.cpu_islands // WORLD if info.gpu_count else 1
        gpu_devices = [dev] * max(1, n_gpu_isl)
        if not info.gpu_count:
            n_gpu_isl, gpu_devices = 0, []
        gpu_budget = plan.gpu_mem_budget_mb or plan.cpu_mem_budget_mb
        cpu_budget = plan.cpu_mem_budget_mb
    else:
        n_gpu_isl = plan.gpu_islands
        n_cpu_isl = plan.cpu_islands
        gpu_devices = [f"cuda:{i % max(1, info.gpu_count)}"
                       for i in range(n_gpu_isl)]
        gpu_budget = plan.gpu_mem_budget_mb
        cpu_budget = plan.cpu_mem_budget_mb
    total_isl = max(1, n_gpu_isl + n_cpu_isl)
    if args.mem_budget_mb:                      # explicit override wins
        gpu_budget = cpu_budget = args.mem_budget_mb

    if args.calibrate and main_proc:
        devs = sorted(set(gpu_devices)) + (["cpu"] if n_cpu_isl else [])
        log("calibrating device throughput...")
        for d, sp in calibrate(devs).items():
            log(f"    {d:<9} {sp:>8.1f} steps/s")
        import gc
        gc.collect()
        if info.cuda_available:
            torch.cuda.empty_cache()
        log("")

    # An RLIMIT_AS cap is a cheap OOM guard on small CPU-only boxes, but it is
    # actively harmful once CUDA is present: the driver reserves tens of GB of
    # *virtual* address space per context, so any sane-looking cap turns into a
    # spurious "OSError: [Errno 12] Cannot allocate memory" on every worker.
    rlimit_mb = 0 if info.cuda_available else (
        int(info.avail_ram_gb * 1024 * 1.6 / max(1, total_isl)) + 1200)
    on_device = plan.data_placement == "vram"
    placement = "ram" if plan.data_placement == "ram" else "mmap"
    headroom = info.avail_ram_gb * 1024 - dataset_mb() - 380 * (total_isl + 1)
    if placement == "ram" and not IS_DIST and n_cpu_isl and headroom > 0:
        load_cache("ram")            # preload so forked islands share via COW
        log(f"dataset preloaded into RAM, shared copy-on-write by "
            f"{n_cpu_isl} CPU island(s)")
    elif placement == "ram":
        placement = "mmap"           # too tight - fall back to the page cache
        log("RAM too tight for a resident copy -> memory-mapping instead")

    import multiprocessing as mp
    ctx_gpu = mp.get_context("spawn" if info.gpu_count else "fork")
    # forking a process that already holds a CUDA context is unsafe
    ctx_cpu = mp.get_context("spawn" if info.cuda_available else "fork")

    # ---- resume or cold start ---------------------------------------------- #
    st = None if args.fresh else load_state()
    if st:
        rng = random.Random()
        rng.setstate(pickle.loads(bytes.fromhex(st["rng"])))
        gen0, population = st["generation"], st["population"]
        history, best = st["history"], st["best"]
        elapsed_prev = st["elapsed_seconds"]
        log(f"RESUMING from generation {gen0} "
            f"({elapsed_prev/3600:.2f}h already spent, "
            f"best fitness {best['fitness']:.5f})\n")
    else:
        rng = random.Random(args.seed)
        gen0, history, best, elapsed_prev = 0, [], None, 0.0
        population = [random_genome(rng) for _ in range(args.pop_size)]
        log(f"COLD START: population {args.pop_size}\n")

    budget = args.max_hours * 3600
    t_start = time.time()

    def spent():
        return elapsed_prev + (time.time() - t_start)

    log(f"evolution: pop {args.pop_size} | {args.inner_steps} steps/individual | "
        f"elite {args.elite_frac:.0%} | mutate {args.mutate_rate:.0%}"
        f"@sigma {args.mutate_sigma} | immigrants {args.immigrant_frac:.0%}")
    log(f"islands  : {n_gpu_isl} GPU + {n_cpu_isl} CPU = {total_isl} concurrent"
        f" | budget {args.max_hours}h\n")

    gen = gen0
    dead_streak = 0
    while gen < args.generations:
        if spent() > budget:
            log(f"\n*** wall-clock budget reached ({args.max_hours}h) ***")
            break
        if _STOP["flag"]:
            break

        gt = time.time()
        remaining = budget - spent()
        # cap a generation so a slow CPU island can never stall the run
        if history:
            prev = max(h["seconds"] for h in history[-3:])
            cap = max(120.0, args.gen_timeout_mult * prev)
        else:
            cap = remaining
        deadline = time.time() + max(30.0, min(remaining, cap))

        tasks = []
        for i, g in enumerate(population):
            k = genome_key(g)
            init = CKPT / "elites" / f"{k}.pt"
            tasks.append({
                "genome": g,
                "seed": args.seed + gen * 1000 + i,
                "inner_steps": args.inner_steps,
                "parsimony": args.parsimony,
                "deadline": deadline,
                "init_state": str(init) if (args.lamarckian and init.exists()) else None,
                "save_state": str(CKPT / "elites" / f"{k}.pt"),
                "rlimit_mb": rlimit_mb,
                "placement": placement,
            })

        def bind_devices(subset):
            """Cheapest genomes go to CPU islands; the rest round-robin the GPUs."""
            n = len(subset)
            order = sorted(range(n), key=lambda i: cost_proxy(
                subset[i]["genome"], meta["n_features"]))
            if not gpu_devices:
                cpu_set = set(range(n))
            elif n_cpu_isl:
                cpu_set = set(order[:min(n_cpu_isl, n)])
            else:
                cpu_set = set()
            for i, t in enumerate(subset):
                to_cpu = i in cpu_set
                t["device"] = "cpu" if to_cpu else gpu_devices[i % len(gpu_devices)]
                t["threads"] = plan.threads_per_cpu_island if to_cpu else 1
                t["mem_budget_mb"] = cpu_budget if to_cpu else gpu_budget
                t["precision"] = "fp32" if to_cpu else plan.precision
                t["on_device"] = on_device and not to_cpu
            return subset

        if IS_DIST:
            import torch.distributed as dist
            if args.parallel == "ddp":
                for t in tasks:
                    t["device"] = gpu_devices[0] if gpu_devices else "cpu"
                    t["threads"] = 1
                    t["mem_budget_mb"] = gpu_budget
                    t["precision"] = plan.precision
                    t["on_device"] = on_device
                scored = [ddp_evaluate(t, RANK, WORLD) for t in tasks]
            else:
                mine = evaluate_population(bind_devices(tasks[RANK::WORLD]),
                                           n_gpu_isl, n_cpu_isl, ctx_gpu, ctx_cpu)
                bucket = [None] * WORLD
                dist.all_gather_object(bucket, mine)
                scored = [r for part in bucket for r in part]
        else:
            scored = evaluate_population(bind_devices(tasks), n_gpu_isl,
                                         n_cpu_isl, ctx_gpu, ctx_cpu)

        scored.sort(key=lambda d: d["fitness"])
        # a truncated individual trained on a short budget - don't crown it
        champ = next((s for s in scored if not s.get("truncated")), scored[0])

        if main_proc and champ.get("state_path") and (
                best is None or champ["fitness"] < best["fitness"]):
            best = dict(champ)
            src_p = Path(champ["state_path"])
            if src_p.exists():
                import shutil
                tmp = CKPT / "best_model.pt.tmp"
                shutil.copyfile(src_p, tmp)
                os.replace(tmp, CKPT / "best_model.pt")
                (CKPT / "best_meta.json").write_text(json.dumps({
                    "genome": best["genome"], "fitness": best["fitness"],
                    "mean_skill": best["mean_skill"],
                    "skill_per_h": best["skill_per_h"], "dir_acc": best["dir_acc"],
                    "params": best["params"],
                    "horizon_names": meta["horizon_names"],
                    "horizon_steps": meta["horizon_steps"],
                    "feature_names": meta["feature_names"],
                    "generation": gen,
                }, indent=2))

        ok = [s for s in scored if s["fitness"] < 1e5]
        n_fail = len(scored) - len(ok)
        n_trunc = sum(1 for s in scored if s.get("truncated") and not s.get("error"))
        history.append({
            "generation": gen, "best_fitness": champ["fitness"],
            "mean_fitness": float(np.mean([s["fitness"] for s in ok])) if ok
            else float("nan"),
            "best_dir_acc": champ["mean_dir_acc"],
            "best_backbone": champ["genome"]["backbone"],
            "n_failed": n_fail, "n_truncated": n_trunc,
            "seconds": round(time.time() - gt, 1),
        })

        log(f"gen {gen:>3} | best {champ['fitness']:.5f} "
            f"({champ['genome']['backbone']}, {champ['params']:,}p, "
            f"dir {champ['mean_dir_acc']*100:.2f}%) | "
            f"mean {history[-1]['mean_fitness']:.5f} | "
            f"fail {n_fail} trunc {n_trunc} | "
            f"{history[-1]['seconds']:.0f}s | "
            f"elapsed {spent()/3600:.2f}h / {args.max_hours}h")
        for s_ in scored:
            if s_.get("error"):
                log(f"        ! [{s_.get('device','?')}] "
                    f"{s_['genome']['backbone']:<5} {s_['error']}")
        if n_fail == len(scored):
            dead_streak += 1
            log(f"        !! entire generation failed "
                f"({dead_streak}/{args.max_dead_generations})")
            if dead_streak >= args.max_dead_generations:
                errs = {}
                for s_ in scored:
                    e = str(s_.get("error", "?")).split(":")[0]
                    errs[e] = errs.get(e, 0) + 1
                log("\n" + "!" * 70)
                log("ABORTING: every individual failed "
                    f"{dead_streak} generations in a row. Not burning the "
                    "remaining budget.")
                log(f"  dominant errors: {errs}")
                if any("Cannot allocate memory" in str(s_.get("error", ""))
                       for s_ in scored):
                    log("")
                    log("  ENOMEM with free RAM almost always means one of:")
                    log("    * the disk is full          -> df -h")
                    log("    * /dev/shm is tiny          -> df -h /dev/shm  "
                        "(needs GBs; docker --shm-size=16g)")
                    log("    * RLIMIT_AS / RLIMIT_DATA caps virtual memory")
                    log("    * vm.max_map_count or vm.overcommit_memory=2")
                log("")
                log("  Diagnose:  python3 scripts/diagnose.py")
                log("  Retry small: ./scripts/run_multigpu.sh "
                    "--islands-per-gpu 1 --no-cpu-islands")
                log("  Your last good checkpoint is intact and will resume.")
                log("!" * 70)
                break
        else:
            dead_streak = 0

        population = next_generation(
            [{"genome": s_["genome"], "fitness": s_["fitness"]} for s_ in scored],
            args.pop_size, rng, args.elite_frac, args.mutate_rate,
            args.immigrant_frac, args.mutate_sigma)
        gen += 1

        if IS_DIST:
            import torch.distributed as dist
            box = [population]
            dist.broadcast_object_list(box, src=0)
            population = box[0]

        if main_proc:
            save_state({
                "generation": gen, "population": population, "history": history,
                "best": best, "elapsed_seconds": spent(),
                "rng": pickle.dumps(rng.getstate()).hex(), "args": vars(args),
                "meta": meta, "strategy": info.strategy,
                "nvlink": info.nvlink_available, "world_size": WORLD,
                "plan": {"gpu_islands": n_gpu_isl, "cpu_islands": n_cpu_isl,
                         "precision": plan.precision,
                         "data_placement": plan.data_placement},
                "parallel": args.parallel if IS_DIST else "islands",
            })
            keep = {genome_key(g) for g in population}
            if best:
                keep.add(best["key"])
            for f in (CKPT / "elites").glob("*.pt"):
                if f.stem not in keep:
                    f.unlink(missing_ok=True)

    if main_proc:
        save_state({
            "generation": gen, "population": population, "history": history,
            "best": best, "elapsed_seconds": spent(),
            "rng": pickle.dumps(rng.getstate()).hex(), "args": vars(args),
            "meta": meta, "strategy": info.strategy,
            "nvlink": info.nvlink_available, "world_size": WORLD,
            "plan": {"gpu_islands": n_gpu_isl, "cpu_islands": n_cpu_isl,
                     "precision": plan.precision,
                     "data_placement": plan.data_placement},
            "parallel": args.parallel if IS_DIST else "islands",
        })
        log("\n" + "=" * 70)
        if best:
            log(f"BEST fitness   : {best['fitness']:.5f}  (1.0 = zero-return baseline)")
            log(f"     backbone  : {best['genome']['backbone']}  "
                f"{best['params']:,} params")
            log(f"     dir. acc  : {best['mean_dir_acc']*100:.2f}% mean")
            for nm, sk, da in zip(meta["horizon_names"], best["skill_per_h"],
                                  best["dir_acc"]):
                log(f"       {nm:<7} skill {sk:.4f}   dir {da*100:5.2f}%")
        log(f"generations    : {gen}")
        log(f"elapsed        : {spent()/3600:.3f} h")
        log(f"checkpoint     : {CKPT/'state.json'}")
        log(f"model          : {CKPT/'best_model.pt'}")
        log("=" * 70)

    if IS_DIST:
        import torch.distributed as dist
        dist.barrier()
        dist.destroy_process_group()

### Launch

`max_hours=5` is the requested budget. Re-run this cell after an interruption
and it picks up from the last completed generation — set `fresh=True` to start
over.

> **Multi-GPU:** run it from the shell instead, so `torchrun` can spawn one
> rank per GPU and NCCL can use NVLink:
> ```bash
> ./scripts/run_multigpu.sh                 # population-parallel, 1 island/GPU
> ./scripts/run_multigpu.sh --parallel ddp  # all GPUs on one individual
> ```
> Notebook kernels can't be `spawn`-pickled, so in-notebook GPU runs use a
> single worker.

In [ ]:
args = argparse.Namespace(
    # --- budget -------------------------------------------------------
    generations=1000,
    pop_size=24,
    inner_steps=2000,         # more training per individual
    max_hours=5.0,            # <- the 5-hour budget; carried across resumes
    # --- evolution rate: deliberately LOW ------------------------------
    elite_frac=0.30,          # strong elitism
    mutate_rate=0.12,         # gentle mutation
    mutate_sigma=0.15,
    immigrant_frac=0.04,      # little random churn
    parsimony=0.002,
    # --- hardware: use everything --------------------------------------
    islands_per_gpu=4,        # several islands per GPU when VRAM allows
    cpu_islands=-1,           # -1 = auto from spare cores, 0 = disable
    no_cpu_islands=False,
    mem_budget_mb=0,          # 0 = auto from detected RAM/VRAM
    parallel="population",
    calibrate=True,
    gen_timeout_mult=2.5,
    # --- misc -----------------------------------------------------------
    subsample=None,           # e.g. 120_000 for a fast smoke test
    seed=1337,
    lamarckian=True,
    fresh=False,              # True = ignore any existing checkpoint
    rebuild_cache=False,
)
run(args)

### Evolution history

In [ ]:
_st = json.loads((CKPT / "state.json").read_text())
hist = pd.DataFrame(_st["history"])
print(f"generations completed : {_st['generation']}")
print(f"wall-clock spent      : {_st['elapsed_seconds']/3600:.3f} h")
print(f"best fitness          : {_st['best']['fitness']:.5f}")

if len(hist):
    ax = hist.plot(x="generation", y=["best_fitness", "mean_fitness"],
                   figsize=(9, 4), grid=True,
                   title="Evolution progress (lower = better)")
    ax.axhline(1.0, color="crimson", ls="--", lw=1,
               label="zero-return baseline")
    ax.set_ylabel("skill  (MSE / baseline MSE)")
    ax.legend()
hist.tail(10)

## 6 · Held-out evaluation

The test window is the most recent stretch of the year and was never touched
during evolution.

* **skill / R²-vs-0** — beat "assume no change"?
* **dir_acc** — directional accuracy; the number that actually matters
* **IC** — Spearman rank correlation of prediction vs outcome

In [ ]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import torch


ROOT = ROOT


def _spearman(a: np.ndarray, b: np.ndarray) -> float:
    ra = np.argsort(np.argsort(a)).astype(np.float64)
    rb = np.argsort(np.argsort(b)).astype(np.float64)
    ra -= ra.mean(); rb -= rb.mean()
    d = np.sqrt((ra ** 2).sum() * (rb ** 2).sum())
    return float((ra * rb).sum() / d) if d > 0 else 0.0


def evaluate_split(split: str = "test") -> dict:
    meta = json.loads((CKPT / "best_meta.json").read_text())
    g = meta["genome"]
    D = load_cache()
    X, Y = D["X"], D["Y"]
    idx = D[f"{split}_idx"]
    ystd = D["y_std"]
    n_feat, n_out = D["meta"]["n_features"], D["meta"]["n_horizons"]
    names = D["meta"]["horizon_names"]
    steps = D["meta"]["horizon_steps"]

    model = build_model(g, n_feat, n_out)
    model.load_state_dict(torch.load(CKPT / "best_model.pt", map_location="cpu",
                                     weights_only=True))
    model.eval()

    L = g["lookback"]
    idx = idx[idx >= L - 1]
    offsets = np.arange(-L + 1, 1, dtype=np.int64)

    preds, acts = [], []
    with torch.no_grad():
        bs = 2048
        for i in range(0, len(idx), bs):
            b = idx[i:i + bs]
            win = b[:, None] + offsets[None, :]
            xb = torch.from_numpy(np.ascontiguousarray(X[win]))
            preds.append(model(xb).numpy())
            acts.append(Y[b])
    P = np.concatenate(preds) * ystd          # de-standardise -> log returns
    A = np.concatenate(acts)

    close = np.load(CACHE / "close.npy")
    c0 = close[idx]

    rows = []
    for j, (nm, st) in enumerate(zip(names, steps)):
        p, a = P[:, j], A[:, j]
        mse, base = float(np.mean((p - a) ** 2)), float(np.mean(a ** 2))
        mask = a != 0
        rows.append({
            "horizon": nm.replace("y_", ""),
            "minutes": st,
            "skill": mse / max(base, 1e-15),
            "r2_vs_zero": 1 - mse / max(base, 1e-15),
            "dir_acc": float(np.mean(np.sign(p[mask]) == np.sign(a[mask]))),
            "ic": _spearman(p, a),
            "mae_bps": float(np.mean(np.abs(p - a))) * 1e4,
            "price_mape": float(np.mean(np.abs(c0 * np.expm1(p) - c0 * np.expm1(a))
                                        / (c0 * np.exp(a)))) * 100,
        })

    print("=" * 84)
    print(f"  HELD-OUT {split.upper()} SET  -  {len(idx):,} samples, "
          f"{g['backbone']} champion")
    print("=" * 84)
    print(f"{'horizon':>8} {'skill':>9} {'R2_vs_0':>9} {'dir_acc':>9} "
          f"{'IC':>9} {'MAE_bps':>10} {'price_MAPE%':>12}")
    print("-" * 84)
    for r in rows:
        print(f"{r['horizon']:>8} {r['skill']:>9.4f} {r['r2_vs_zero']:>9.4f} "
              f"{r['dir_acc']*100:>8.2f}% {r['ic']:>9.4f} {r['mae_bps']:>10.2f} "
              f"{r['price_mape']:>11.4f}%")
    print("-" * 84)
    print(f"{'MEAN':>8} {np.mean([r['skill'] for r in rows]):>9.4f} "
          f"{np.mean([r['r2_vs_zero'] for r in rows]):>9.4f} "
          f"{np.mean([r['dir_acc'] for r in rows])*100:>8.2f}% "
          f"{np.mean([r['ic'] for r in rows]):>9.4f}")
    print("=" * 84)
    print("skill  < 1.0  and  R2_vs_0 > 0  mean the model beats 'predict no change'.")
    print("dir_acc materially > 50% is the signal that actually matters.")

    out = {"split": split, "n_samples": int(len(idx)), "genome": g, "rows": rows}
    (ROOT / "logs" / f"eval_{split}.json").write_text(json.dumps(out, indent=2))
    return out

In [ ]:
_res = evaluate_split("test")
pd.DataFrame(_res["rows"]).set_index("horizon")

## 7 · Forecast

Loads the evolved champion and projects BTC forward from the newest bar.
Bands are ±1 MAE-derived sigma from the held-out evaluation.

In [ ]:
import argparse
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch


ROOT = ROOT
CKPT = ROOT / "checkpoints"
CACHE = ROOT / "data" / "processed" / "cache"
DATA = ROOT / "data" / "processed" / "btc_1m.parquet"


def load_champion():
    meta = json.loads((CKPT / "best_meta.json").read_text())
    g = meta["genome"]
    n_feat = len(meta["feature_names"])
    n_out = len(meta["horizon_names"])
    model = build_model(g, n_feat, n_out)
    model.load_state_dict(torch.load(CKPT / "best_model.pt", map_location="cpu",
                                     weights_only=True))
    model.eval()
    if not (CACHE / "scaler.pkl").exists():
        # regenerable, so not committed: build it on first use
        print("building the feature cache (one-off, ~20 s)...", flush=True)
        build_cache()
    with open(CACHE / "scaler.pkl", "rb") as f:
        scaler = pickle.load(f)
    y_std = np.load(CACHE / "y_std.npy")
    return model, meta, scaler, y_std


def forecast(df: pd.DataFrame | None = None) -> dict:
    model, meta, scaler, y_std = load_champion()
    g = meta["genome"]
    L = g["lookback"]

    if df is None:
        df = pd.read_parquet(DATA)
    need = L + 1500                                   # feature warm-up history
    df = df.iloc[-need:].reset_index(drop=True)

    F = build_features(df)
    F = F[meta["feature_names"]]                      # exact training order
    Xn = np.clip((F.values.astype(np.float32) - scaler["median"]) / scaler["scale"],
                 -8, 8)
    Xn = np.nan_to_num(Xn, nan=0.0).astype(np.float32)

    win = torch.from_numpy(Xn[-L:][None, ...])        # (1, L, F)
    with torch.no_grad():
        logret = model(win).numpy()[0] * y_std

    spot = float(df.close.iloc[-1])
    ts = pd.Timestamp(df.ts.iloc[-1])

    # residual sigma per horizon, taken from the held-out evaluation if present
    sig = None
    ev = ROOT / "logs" / "eval_test.json"
    if ev.exists():
        e = json.loads(ev.read_text())
        sig = {r["horizon"]: r["mae_bps"] / 1e4 * 1.2533 for r in e["rows"]}

    out = []
    for nm, lr in zip(meta["horizon_names"], logret):
        h = nm.replace("y_", "")
        price = spot * float(np.exp(lr))
        row = {
            "horizon": h,
            "minutes": HORIZONS[h],
            "valid_at": str(ts + pd.Timedelta(minutes=HORIZONS[h])),
            "pred_log_return": float(lr),
            "pred_return_pct": float(np.expm1(lr) * 100),
            "pred_price": price,
            "direction": "UP" if lr > 0 else "DOWN",
        }
        if sig and h in sig:
            s = sig[h]
            row["price_lo_68"] = spot * float(np.exp(lr - s))
            row["price_hi_68"] = spot * float(np.exp(lr + s))
        out.append(row)

    return {"as_of": str(ts), "spot": spot,
            "backbone": g["backbone"], "forecasts": out}

In [ ]:
fc = forecast()
print(f"as of {fc['as_of']}   spot ${fc['spot']:,.2f}   model: evolved {fc['backbone']}")
pd.DataFrame(fc["forecasts"])[
    ["horizon", "valid_at", "pred_return_pct", "pred_price", "direction"]
].set_index("horizon")

## 8 · Publish to GitHub

The token is read from the `GITHUB_TOKEN` environment variable. **Never paste
a token into a notebook cell** — the value gets saved inside the `.ipynb` and
travels with every copy of it.

```bash
export GITHUB_TOKEN=github_pat_xxxxxxxx
python src/upload_to_github.py --repo btc-price-predictor --mode release
```

In [ ]:
if os.environ.get("GITHUB_TOKEN"):
    import subprocess
    r = subprocess.run(
        [sys.executable, str(ROOT / "src" / "upload_to_github.py"),
         "--repo", "btc-price-predictor", "--mode", "release"],
        cwd=ROOT, text=True, capture_output=True)
    print(r.stdout or "", r.stderr or "")
else:
    print("GITHUB_TOKEN not set - skipping upload.")
    print("Run:  export GITHUB_TOKEN=...  in your shell, restart the kernel,")
    print("then re-run this cell. Do not hard-code the token here.")